# Extração IBGE — População e territorialidade · Conecta Saúde

O IBGE é o **denominador populacional** do Índice Composto de Pressão Assistencial (ICPA). Sem população, um ranking absoluto de internações apenas reordena os municípios por tamanho: São Paulo sempre aparece em primeiro, e o município pequeno com rede saturada some da lista.

**Validação do join com o CNES.** É o cruzamento que sustenta o índice, e ele agora é conferido com dados reais em vez de assumido.

## 1. Papel do IBGE no projeto

O IBGE entra como contexto populacional e territorial:

- população municipal, para normalizar todos os indicadores;
- hierarquia território → município → UF → região, para os filtros do dashboard;
- código IBGE do município, que é a chave de junção com o DATASUS.

Cruzamentos previstos:

| Cruzamento | Indicador resultante |
|---|---|
| IBGE × CNES | Leitos SUS por 10 mil habitantes |
| IBGE × SIH/SUS | Internações por 10 mil habitantes |

## 2. APIs públicas

### Localidades — a estrutura territorial

```text
https://servicodados.ibge.gov.br/api/v1/localidades/municipios
```

### Agregados/SIDRA — a população

```text
https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/{ANO}/variaveis/9324?localidades=N6[all]
```

### Sobre a disponibilidade dos anos

Nem todo ano existe nessa tabela, e a falha é silenciosa: a API responde 200 com a série vazia. Testando os anos diretamente:

| Ano | Localidades com valor |
|---|---|
| 2021 | 5.570 |
| **2022** | **0 — ano censitário, sem estimativa nesta tabela** |
| 2024 | 5.570 |
| 2025 | 5.571 |


## 3. Parâmetros da extração

In [1]:
import gzip
import json
import urllib.request
from pathlib import Path

import pandas as pd

ANO_POPULACAO = 2024

BASE_LOCALIDADES = "https://servicodados.ibge.gov.br/api/v1/localidades"
BASE_AGREGADOS = "https://servicodados.ibge.gov.br/api/v3/agregados"

# Tabela 6579 = População residente estimada; variável 9324 = população residente.
TABELA_POPULACAO = "6579"
VARIAVEL_POPULACAO = "9324"


def raiz_do_projeto() -> Path:
    '''Devolve a pasta do repositório, subindo até encontrar o .git.

    Sem isto os caminhos dependeriam de onde o Jupyter foi aberto: rodando a
    partir de "Fontes de dados" os dados cairiam dentro dessa pasta, e a
    partir da raiz cairiam em outro lugar.
    '''
    atual = Path.cwd().resolve()
    for pasta in (atual, *atual.parents):
        if (pasta / ".git").exists():
            return pasta
    return atual


RAIZ = raiz_do_projeto()
DIRETORIO_RAW = RAIZ / "dados" / "raw" / "ibge"
DIRETORIO_TRATADO = RAIZ / "dados" / "tratado" / "ibge"

for pasta in (DIRETORIO_RAW, DIRETORIO_TRATADO):
    pasta.mkdir(parents=True, exist_ok=True)

print(f"Raiz do projeto : {RAIZ}")
print(f"Saída tratada   : {DIRETORIO_TRATADO}")
print(f"Ano de população: {ANO_POPULACAO}")

Raiz do projeto : C:\Users\Vitor Nobre\Documents\Workspace\conecta-saude
Saída tratada   : C:\Users\Vitor Nobre\Documents\Workspace\conecta-saude\dados\tratado\ibge
Ano de população: 2024


## 4. Conexão e conferência do ano

O `obter_json` trata duas coisas que o `urllib` cru não trata: a resposta em **gzip** e a ausência de `User-Agent`, que alguns endpoints do IBGE recusam.

In [2]:
def obter_json(url: str, timeout: int = 300):
    '''Baixa e decodifica JSON da API do IBGE, tratando resposta comprimida.'''
    requisicao = urllib.request.Request(
        url,
        headers={"User-Agent": "conecta-saude/1.0", "Accept-Encoding": "gzip"},
    )
    with urllib.request.urlopen(requisicao, timeout=timeout) as resposta:
        bruto = resposta.read()

    if bruto[:2] == b"\x1f\x8b":
        bruto = gzip.decompress(bruto)

    return json.loads(bruto.decode("utf-8"))


def url_populacao(ano: int) -> str:
    return (
        f"{BASE_AGREGADOS}/{TABELA_POPULACAO}/periodos/{ano}"
        f"/variaveis/{VARIAVEL_POPULACAO}?localidades=N6[all]"
    )


def contar_localidades_com_valor(payload: list, ano: int) -> int:
    '''Conta quantos municípios têm população preenchida no ano pedido.'''
    if not payload:
        return 0
    series = payload[0].get("resultados", [{}])[0].get("series", [])
    return sum(
        1 for s in series
        if s.get("serie", {}).get(str(ano)) not in (None, "", "...", "-")
    )


pop_json = obter_json(url_populacao(ANO_POPULACAO))
com_valor = contar_localidades_com_valor(pop_json, ANO_POPULACAO)
print(f"Tabela {TABELA_POPULACAO}, ano {ANO_POPULACAO}: {com_valor:,} municípios com população")

assert com_valor > 5000, (
    f"O ano {ANO_POPULACAO} não tem estimativa publicada na tabela {TABELA_POPULACAO} "
    f"(apenas {com_valor} municípios com valor). Anos censitários como 2022 ficam vazios "
    "nesta tabela — escolha outro ano."
)

Tabela 6579, ano 2024: 5,570 municípios com população


## 5. Hierarquia territorial dos municípios

O endpoint de localidades devolve o município com toda a hierarquia aninhada — microrregião dentro de mesorregião dentro de UF dentro de região. A função abaixo achata isso numa tabela por município.

In [3]:
def normalizar_municipios(payload: list[dict]) -> pd.DataFrame:
    '''Achata o JSON hierárquico do IBGE numa tabela por município.'''
    linhas = []
    for item in payload:
        microrregiao = item.get("microrregiao") or {}
        mesorregiao = microrregiao.get("mesorregiao") or {}
        uf = mesorregiao.get("UF") or {}
        regiao = uf.get("regiao") or {}
        imediata = item.get("regiao-imediata") or {}
        intermediaria = imediata.get("regiao-intermediaria") or {}

        linhas.append({
            "cod_municipio_ibge": str(item.get("id")),
            "municipio": item.get("nome"),
            "cod_uf": uf.get("id"),
            "uf": uf.get("sigla"),
            "estado": uf.get("nome"),
            "cod_regiao": regiao.get("id"),
            "regiao": regiao.get("nome"),
            "cod_microrregiao": microrregiao.get("id"),
            "microrregiao": microrregiao.get("nome"),
            "cod_mesorregiao": mesorregiao.get("id"),
            "mesorregiao": mesorregiao.get("nome"),
            "cod_regiao_imediata": imediata.get("id"),
            "regiao_imediata": imediata.get("nome"),
            "cod_regiao_intermediaria": intermediaria.get("id"),
            "regiao_intermediaria": intermediaria.get("nome"),
        })
    return pd.DataFrame(linhas)


municipios_json = obter_json(f"{BASE_LOCALIDADES}/municipios")
df_municipios = normalizar_municipios(municipios_json)

print(f"Municípios: {len(df_municipios):,} | UFs: {df_municipios['uf'].nunique()}")
df_municipios.head()

Municípios: 5,571 | UFs: 27


,cod_municipio_ibge,municipio,cod_uf,uf,estado,cod_regiao,regiao,cod_microrregiao,microrregiao,cod_mesorregiao,mesorregiao,cod_regiao_imediata,regiao_imediata,cod_regiao_intermediaria,regiao_intermediaria
0,1100015,Alta Floresta D'Oeste,11.0,RO,Rondônia,1.0,Norte,11006.0,Cacoal,1102.0,Leste Rondoniense,110005,Cacoal,1102,Ji-Paraná
1,1100023,Ariquemes,11.0,RO,Rondônia,1.0,Norte,11003.0,Ariquemes,1102.0,Leste Rondoniense,110002,Ariquemes,1101,Porto Velho
2,1100031,Cabixi,11.0,RO,Rondônia,1.0,Norte,11008.0,Colorado do Oeste,1102.0,Leste Rondoniense,110006,Vilhena,1102,Ji-Paraná
3,1100049,Cacoal,11.0,RO,Rondônia,1.0,Norte,11006.0,Cacoal,1102.0,Leste Rondoniense,110005,Cacoal,1102,Ji-Paraná
4,1100056,Cerejeiras,11.0,RO,Rondônia,1.0,Norte,11008.0,Colorado do Oeste,1102.0,Leste Rondoniense,110006,Vilhena,1102,Ji-Paraná


## 6. População municipal

A resposta da API de Agregados é aninhada em quatro níveis — variável, resultado, série, localidade. A função abaixo desce até o valor do ano pedido.

In [4]:
def normalizar_populacao(payload: list[dict], ano: int) -> pd.DataFrame:
    '''Extrai código do município e população do JSON aninhado da API de Agregados.'''
    linhas = []
    for bloco in payload:
        for resultado in bloco.get("resultados", []):
            for serie in resultado.get("series", []):
                localidade = serie.get("localidade", {})
                valor = serie.get("serie", {}).get(str(ano))
                linhas.append({
                    "cod_municipio_ibge": str(localidade.get("id")),
                    "ano_populacao": ano,
                    "populacao": pd.to_numeric(valor, errors="coerce"),
                })
    return pd.DataFrame(linhas)


df_populacao = normalizar_populacao(pop_json, ANO_POPULACAO)

print(f"Registros de população: {len(df_populacao):,}")
print(f"Sem valor numérico    : {int(df_populacao['populacao'].isna().sum())}")
print(f"População total       : {int(df_populacao['populacao'].sum()):,}")
df_populacao.head()

Registros de população: 5,571
Sem valor numérico    : 1
População total       : 212,583,750


,cod_municipio_ibge,ano_populacao,populacao
0,1100015,2024,22853.0
1,1100023,2024,108573.0
2,1100031,2024,5690.0
3,1100049,2024,97637.0
4,1100056,2024,16975.0


## 7. Base final e a chave de junção com o DATASUS

Aqui está o detalhe que faz ou quebra todo o cruzamento do projeto.

O IBGE identifica município com **7 dígitos** (`3550308` = São Paulo). O DATASUS usa **6** — os mesmos dígitos, sem o último, que é o verificador. O `CODUFMUN` do CNES e o `MUNIC_RES` do SIH vêm nesse formato de 6.

Cortar o sétimo dígito é seguro porque os 6 primeiros já identificam o município de forma única: são 5.571 códigos de 7 dígitos e 5.571 códigos de 6 dígitos distintos, sem nenhuma colisão. A célula confere isso em vez de confiar.

In [5]:
df_ibge = df_municipios.merge(
    df_populacao[["cod_municipio_ibge", "ano_populacao", "populacao"]],
    on="cod_municipio_ibge",
    how="left",
)

# Chave de 6 dígitos usada pelo DATASUS (CODUFMUN no CNES, MUNIC_RES no SIH).
df_ibge["cod_municipio_datasus"] = df_ibge["cod_municipio_ibge"].str[:6]

# Cortar o dígito verificador só é seguro se não gerar colisão.
assert df_ibge["cod_municipio_datasus"].nunique() == df_ibge["cod_municipio_ibge"].nunique(), (
    "o corte para 6 dígitos criou códigos duplicados — o join com o DATASUS ficaria ambíguo"
)

print(f"Municípios: {len(df_ibge):,}")
print(f"Sem população: {int(df_ibge['populacao'].isna().sum())}")
print(f"Chave de 6 dígitos sem colisão: {df_ibge['cod_municipio_datasus'].nunique():,} códigos distintos")
df_ibge.head()

Municípios: 5,571
Sem população: 1
Chave de 6 dígitos sem colisão: 5,571 códigos distintos


,cod_municipio_ibge,municipio,cod_uf,uf,estado,cod_regiao,regiao,cod_microrregiao,microrregiao,cod_mesorregiao,mesorregiao,cod_regiao_imediata,regiao_imediata,cod_regiao_intermediaria,regiao_intermediaria,ano_populacao,populacao,cod_municipio_datasus
0,1100015,Alta Floresta D'Oeste,11.0,RO,Rondônia,1.0,Norte,11006.0,Cacoal,1102.0,Leste Rondoniense,110005,Cacoal,1102,Ji-Paraná,2024,22853.0,110001
1,1100023,Ariquemes,11.0,RO,Rondônia,1.0,Norte,11003.0,Ariquemes,1102.0,Leste Rondoniense,110002,Ariquemes,1101,Porto Velho,2024,108573.0,110002
2,1100031,Cabixi,11.0,RO,Rondônia,1.0,Norte,11008.0,Colorado do Oeste,1102.0,Leste Rondoniense,110006,Vilhena,1102,Ji-Paraná,2024,5690.0,110003
3,1100049,Cacoal,11.0,RO,Rondônia,1.0,Norte,11006.0,Cacoal,1102.0,Leste Rondoniense,110005,Cacoal,1102,Ji-Paraná,2024,97637.0,110004
4,1100056,Cerejeiras,11.0,RO,Rondônia,1.0,Norte,11008.0,Colorado do Oeste,1102.0,Leste Rondoniense,110006,Vilhena,1102,Ji-Paraná,2024,16975.0,110005


## 8. Validação contra o CNES

O join IBGE × DATASUS é a base de todo indicador por habitante do projeto, então vale conferir com dado real em vez de assumir que funciona.

A célula abaixo roda apenas se o notebook do CNES já tiver sido executado. Com os dados de dezembro/2024, o resultado é:

- **3.568 de 3.568** municípios com leito no CNES encontram seu par no IBGE — nenhum órfão;
- **2.003** municípios do IBGE não aparecem no CNES.

O segundo número não é falha de extração: são municípios sem nenhum leito cadastrado. É exatamente o vazio assistencial que o índice precisa enxergar, e é por isso que o cruzamento final tem de partir do IBGE, com `how="left"`. Partindo do CNES, esses 2.003 municípios simplesmente desapareceriam do painel.

In [7]:
caminho_cnes = RAIZ / "dados" / "tratado" / "cnes" / "cnes_silver_municipio_2024.parquet"

if caminho_cnes.exists():
    cnes = pd.read_parquet(caminho_cnes)
    ultima = cnes["COMPETENCIA"].max()
    cnes = cnes[cnes["COMPETENCIA"] == ultima]

    conferencia = cnes.merge(
        df_ibge[["cod_municipio_datasus", "municipio", "uf", "populacao"]],
        left_on="CODUFMUN",
        right_on="cod_municipio_datasus",
        how="left",
    )
    orfaos = conferencia[conferencia["municipio"].isna()]

    print(f"Competência {ultima}")
    print(f"  municípios com leito no CNES : {len(cnes):,}")
    print(f"  encontraram par no IBGE      : {int(conferencia['municipio'].notna().sum()):,}")
    print(f"  órfãos (sem par)             : {len(orfaos):,}")
    if not orfaos.empty:
        print(f"  códigos órfãos: {sorted(orfaos['CODUFMUN'].unique())[:20]}")

    sem_leito = ~df_ibge["cod_municipio_datasus"].isin(cnes["CODUFMUN"])
    print(f"\n  municípios do IBGE sem leito no CNES: {int(sem_leito.sum()):,} de {len(df_ibge):,}")
    print(f"  população vivendo neles: {int(df_ibge.loc[sem_leito, 'populacao'].sum()):,}")
else:
    print("Notebook do CNES ainda não foi executado — validação do join pulada.")
    print(f"Esperado em: {caminho_cnes}")

Competência 202412
  municípios com leito no CNES : 3,568
  encontraram par no IBGE      : 3,568
  órfãos (sem par)             : 0

  municípios do IBGE sem leito no CNES: 2,003 de 5,571
  população vivendo neles: 14,487,908


## 9. Validação de qualidade

Estes números vão para o slide de tratamento de dados.

In [8]:
validacoes = pd.DataFrame([
    {"verificacao": "Municípios extraídos", "valor": len(df_ibge)},
    {"verificacao": "UFs distintas", "valor": int(df_ibge["uf"].nunique())},
    {"verificacao": "Regiões distintas", "valor": int(df_ibge["regiao"].nunique())},
    {"verificacao": "Municípios sem população", "valor": int(df_ibge["populacao"].isna().sum())},
    {"verificacao": "Código IBGE fora do padrão de 7 dígitos",
     "valor": int((df_ibge["cod_municipio_ibge"].str.len() != 7).sum())},
    {"verificacao": "Código DATASUS fora do padrão de 6 dígitos",
     "valor": int((df_ibge["cod_municipio_datasus"].str.len() != 6).sum())},
    {"verificacao": "População total do país", "valor": int(df_ibge["populacao"].sum())},
])

display(validacoes)

print("\nPopulação por região:")
display(
    df_ibge.groupby("regiao")
    .agg(municipios=("cod_municipio_ibge", "nunique"), populacao=("populacao", "sum"))
    .sort_values("populacao", ascending=False)
)

,verificacao,valor
0,Municípios extraídos,5571
1,UFs distintas,27
2,Regiões distintas,5
3,Municípios sem população,1
4,Código IBGE fora do padrão de 7 dígitos,0
5,Código DATASUS fora do padrão de 6 dígitos,0
6,População total do país,212583750



População por região:


,municipios,populacao
regiao,,
Sudeste,1668,88617693.0
Nordeste,1794,57112096.0
Sul,1191,31113021.0
Norte,450,18669345.0
Centro-Oeste,467,17071595.0


## 10. Salvamento em Parquet

Saída em `dados/tratado/ibge/`. É uma tabela pequena — 5.571 linhas — e serve como dimensão territorial para todas as outras fontes, então fica em arquivo único.

In [9]:
destino = DIRETORIO_TRATADO / f"ibge_municipios_populacao_{ANO_POPULACAO}.parquet"
df_ibge.to_parquet(destino, index=False)

print(f"{destino.name}  ({destino.stat().st_size / 1024:.1f} KB)")
print(f"Pasta: {DIRETORIO_TRATADO}")
print(f"{len(df_ibge):,} linhas x {df_ibge.shape[1]} colunas")

ibge_municipios_populacao_2024.parquet  (234.9 KB)
Pasta: C:\Users\Vitor Nobre\Documents\Workspace\conecta-saude\dados\tratado\ibge
5,571 linhas x 18 colunas


## 11. Perguntas que o IBGE responde

1. Qual a população de cada município?
2. Quais municípios pertencem a cada UF, região de saúde e região do país?
3. Como comparar municípios de tamanhos diferentes de forma justa?
4. Quantas pessoas vivem em municípios sem nenhum leito SUS?

As fórmulas que o IBGE viabiliza:

```text
Internações por 10 mil hab.   = (internações SIH / população) * 10.000
Leitos SUS por 10 mil hab.    = (leitos SUS CNES / população) * 10.000
Diárias de UTI por 10 mil hab.= (diárias UTI SIH / população) * 10.000
```

A população também é o que permite medir o vazio assistencial em pessoas, e não em municípios: os 2.003 municípios sem nenhum leito SUS abrigam 14,5 milhões de habitantes.

## 12. Exemplo — leitos SUS por 10 mil habitantes

Junta IBGE e CNES e calcula o primeiro indicador normalizado do projeto. Roda só se o CNES já tiver sido extraído.

Repare no `how="left"` a partir do IBGE: municípios sem leito entram com zero, não somem. Um `inner join` aqui produziria um painel que não enxerga justamente os municípios mais desassistidos.

In [10]:
if caminho_cnes.exists():
    base = df_ibge[["cod_municipio_datasus", "municipio", "uf", "regiao", "populacao"]].copy()

    leitos = (
        pd.read_parquet(caminho_cnes)
        .query("COMPETENCIA == @ultima")[["CODUFMUN", "leitos_sus"]]
    )

    # left a partir do IBGE: município sem leito vira 0, e não desaparece.
    indicador = base.merge(
        leitos, left_on="cod_municipio_datasus", right_on="CODUFMUN", how="left"
    )
    indicador["leitos_sus"] = indicador["leitos_sus"].fillna(0).astype(int)
    indicador["leitos_por_10k"] = (
        indicador["leitos_sus"] / indicador["populacao"] * 10_000
    ).round(2)

    print(f"Municípios: {len(indicador):,} | sem nenhum leito SUS: "
          f"{int((indicador['leitos_sus'] == 0).sum()):,}")

    grandes = indicador[indicador["populacao"] >= 100_000]
    print(f"\nEntre os {len(grandes):,} municípios com 100 mil+ habitantes:")
    print("\n  menor oferta de leitos por 10 mil hab.:")
    display(grandes.nsmallest(10, "leitos_por_10k")[
        ["municipio", "uf", "populacao", "leitos_sus", "leitos_por_10k"]
    ])

    print("\n  Por região:")
    display(
        indicador.groupby("regiao")
        .apply(lambda g: pd.Series({
            "populacao": g["populacao"].sum(),
            "leitos_sus": g["leitos_sus"].sum(),
            "leitos_por_10k": round(g["leitos_sus"].sum() / g["populacao"].sum() * 10_000, 2),
            "municipios_sem_leito": int((g["leitos_sus"] == 0).sum()),
        }), include_groups=False)
        .sort_values("leitos_por_10k")
    )
else:
    print("Execute o notebook do CNES primeiro para ver este exemplo.")

Municípios: 5,571 | sem nenhum leito SUS: 2,003

Entre os 336 municípios com 100 mil+ habitantes:

  menor oferta de leitos por 10 mil hab.:


,municipio,uf,populacao,leitos_sus,leitos_por_10k
3550,Jandira,SP,121988.0,0,0.00
3711,Poá,SP,106431.0,0,0.00
3915,Almirante Tamandaré,PR,124788.0,0,0.00
4491,Palhoça,SC,245477.0,0,0.00
1465,Abreu e Lima,PE,103945.0,8,0.77
5564,Valparaíso de Goiás,GO,213506.0,35,1.64
579,Paço do Lumiar,MA,152306.0,26,1.71
1543,Ipojuca,PE,105638.0,20,1.89
1501,Camaragibe,PE,155771.0,32,2.05
3990,Colombo,PR,240720.0,53,2.20



  Por região:


,populacao,leitos_sus,leitos_por_10k,municipios_sem_leito
regiao,,,,
Sudeste,88617693.0,125892.0,14.21,773.0
Norte,18669345.0,30609.0,16.40,117.0
Centro-Oeste,17071595.0,29982.0,17.56,105.0
Sul,31113021.0,57317.0,18.42,563.0
Nordeste,57112096.0,106740.0,18.69,444.0
